In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("aocdml.keras")
last_conv_layer_name = model.get_layer("conv2d_2").name


In [ ]:
import numpy as np

# def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
#     if not model.built:
#         dummy = tf.zeros((1, 224, 224, 1))
#         _ = model(dummy)

#     grad_model = tf.keras.models.Model(
#         inputs=model.inputs,
#         outputs=[
#             model.get_layer(last_conv_layer_name).output,
#             model.outputs
#         ]
#     )

#     with tf.GradientTape() as tape:
#         conv_outputs, predictions = grad_model(img_array)
#         if isinstance(predictions, list):
#             predictions = predictions[0]
        
#         if pred_index is None:
#             pred_index = tf.argmax(predictions[0])
#         class_channel = predictions[:, pred_index]

#     grads = tape.gradient(class_channel, conv_outputs)
#     pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
#     conv_outputs = conv_outputs[0]

#     heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
#     heatmap = tf.squeeze(heatmap)

#     heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
#     return heatmap.numpy()


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):

    # Get last conv layer
    last_conv_layer = model.get_layer(last_conv_layer_name)

    # Create model that outputs conv layer + logits (before softmax)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[last_conv_layer.output, model.layers[-1].output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, logits = grad_model(img_array, training=True)

        if isinstance(logits, list):
            logits = logits[0]

        if pred_index is None:
            pred_index = tf.argmax(logits[0])

        class_channel = logits[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    # If grads is None, stop here
    if grads is None:
        raise ValueError("Gradients are None — check model connectivity.")

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.matmul(conv_outputs, pooled_grads[..., tf.newaxis])
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

In [ ]:
import shap

def make_shap_explanation(img_array, model):
    background_data = np.load("background_data.npy")
    if background_data.ndim == 3:
        background_data = np.expand_dims(background_data, axis=-1)
    if img_array.ndim == 3:
        img_array = np.expand_dims(img_array, axis=0) 
    print("Background data shape:", background_data.shape)
    explainer = shap.GradientExplainer(model, background_data)
    shap_values = explainer.shap_values(img_array)
    return shap_values


In [ ]:
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image


# Loading Image
img = image.load_img("check_XAI/OC_2.JPG", target_size=(224,224), color_mode="grayscale")  # 1 channel
img_array = image.img_to_array(img)  # shape: (224,224,1)
img_array = np.expand_dims(img_array, axis=0)  # shape: (1,224,224,1)

print("Correct img_array shape:", img_array.shape)

# Making Prediction and Classifying Image
prediction = model.predict(img_array)
predicted_class = np.argmax(prediction, axis=1)

# # Overlaying Heatmap on OG Image
# model.predict(img_array)
# heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
# heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
# heatmap = np.uint8(255 * heatmap)
# heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
# img_display = np.uint8(255 * img)
# superimposed_img = cv2.addWeighted(img_display, 0.6, heatmap, 0.4, 0)
# # superimposed_img = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB) # If it looks weird

# Collecting SHAP values
shap_values = make_shap_explanation(img_array, model)

# Returning Outputs
if predicted_class[0] == 0:
    category = "Benign Ovarian Tumor"
elif predicted_class[0] == 1:
    category = "Normal Ovary"
elif predicted_class[0] == 2:
    category = "Ovarian Cancer"
elif predicted_class[0] == 3:
    category = "Uterine Fibroid"
print("Predicted class:", category)
print("Probabilities:", prediction)
print(predicted_class)

# plt.imshow(superimposed_img)
# plt.axis("off")
# plt.title("Grad-CAM")
# plt.show()

shap.image_plot([shap_values[[0]]], img_array)

In [ ]:
print(model.summary())